# Building Footprint Prediction from THEOS-2 Imagery

source code นี้ใช้สำหรับตรวจจับขอบเขตอาคาร (Building Footprint) จากภาพถ่ายดาวเทียม THEOS-2 โดยใช้โมเดล **TerraMind ร่วมกับ Local CNN branch** ที่ผ่านการ fine-tune มาแล้ว

จุดประสงค์ของไฟล์นี้คือทำให้ผู้ใช้งานสามารถเลือกภาพที่ต้องการจาก Google Drive ได้เอง ไม่ว่าจะเป็นภาพเดียวหรือหลายภาพ จากนั้นระบบจะดาวน์โหลด checkpoint เดิมที่ผมเคย fine-tuneไว้ แล้วประมวลผลภาพทั้งหมด และส่งออกผลลัพธ์ทั้งแบบ raster และ vector

## ผลลัพธ์ที่ได้จาก workflow

1. Probability GeoTIFF — ค่าความน่าจะเป็นที่แต่ละ pixel เป็นอาคาร
2. Raw Mask GeoTIFF — mask ที่ได้จากการใช้ threshold
3. Clean Mask GeoTIFF — mask หลังจากทำ post-processing
4. Building Footprint GeoPackage — ไฟล์ polygon อาคารที่สามารถเปิดใน QGIS ได้
5. Summary CSV — ตารางสรุปผลของทุกภาพ

> **แนวทางการใช้งาน:** ให้รันแต่ละ Session จากบนลงล่างตามลำดับ โดยแก้เฉพาะค่าที่ระบุไว้ใน Session 2 และ Session 3


# Session 1 — ติดตั้ง Library ที่จำเป็น

ขั้นตอนนี้ติดตั้ง library สำหรับอ่านข้อมูล GeoTIFF, ประมวลผลข้อมูลเชิงพื้นที่, แสดงผลภาพ และเรียกใช้ TerraMind ผ่าน TerraTorch

มีการกำหนด NumPy เป็นเวอร์ชัน 1.26.4 เพื่อช่วยลดปัญหาความเข้ากันได้ระหว่าง TerraTorch และ library บางตัวใน Google Colab

หลังติดตั้งเสร็จ Colab อาจแจ้งให้ Restart Runtime หากมีการเปลี่ยนเวอร์ชัน package ที่ถูกโหลดอยู่ก่อนแล้ว


In [ ]:
!pip -q install --no-cache-dir "numpy==1.26.4" rasterio geopandas shapely pyproj fiona scipy scikit-image gdown matplotlib tqdm pandas
!pip -q install --no-cache-dir terratorch huggingface_hub

# Session 2 — เชื่อมต่อ Google Drive และกำหนดภาพที่ต้องการใช้

ภาพ THEOS-2 ยังคงอ่านผ่าน **Google Drive Mount (`gmount`)** เพราะผู้ใช้งานสามารถเปลี่ยนจำนวนภาพหรือเลือกโฟลเดอร์ภาพใหม่ได้โดยไม่ต้องแก้ส่วนอื่นของ pipeline

ในตัวแปร `IMAGE_SOURCES` สามารถใส่ได้ทั้ง:

- path ของโฟลเดอร์ ซึ่งระบบจะค้นหาไฟล์ `.tif` และ `.tiff` ภายในโฟลเดอร์นั้น
- path ของไฟล์ GeoTIFF เพียงไฟล์เดียว
- หลาย path พร้อมกัน หากต้องการประมวลผลหลายพื้นที่

ผลลัพธ์จะถูกบันทึกไว้ในพื้นที่ชั่วคราวของ Colab ที่ `/content/outputs_full_valid_rgb` เพื่อไม่ให้สร้างไฟล์จำนวนมากใน Google Drive โดยอัตโนมัติ


In [ ]:
from pathlib import Path
from google.colab import drive

# เชื่อมต่อ Google Drive
drive.mount("/content/drive")

# ============================================================
# แก้ไขเฉพาะส่วนนี้เมื่อต้องการเปลี่ยนภาพ
# ============================================================

IMAGE_SOURCES = [
    "/content/drive/MyDrive/THEOS2_images"
]

# โฟลเดอร์สำหรับเก็บผลลัพธ์ใน Colab
OUTPUT_ROOT = Path("/content/outputs_full_valid_rgb")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_ROOT)
print("Image sources:")
for source in IMAGE_SOURCES:
    print(" -", source)


# Session 3 — ดาวน์โหลด Model Checkpoint ด้วย gdown

Checkpoint จะไม่อ่านจาก path ภายใน Google Drive ของผู้ใช้งานอีกต่อไป แต่จะดาวน์โหลดจาก **Google Drive share link** ด้วย `gdown`

วิธีนี้เหมาะกับการส่งไฟล์ให้ผู้อื่น เพราะผู้รับไม่จำเป็นต้องมีโครงสร้างโฟลเดอร์ checkpoint แบบเดียวกับผู้สร้างโน้ตบุ๊ก

ให้นำลิงก์ Google Drive ของ checkpoint มาใส่ในตัวแปร `CHECKPOINT_URL` โดยตั้งสิทธิ์ของไฟล์เป็น **Anyone with the link**

`gdown` จะใช้ชื่อเดิมของไฟล์จาก Google Drive โดยตรง จึงไม่ต้องบังคับเปลี่ยนชื่อ checkpoint ภายในโค้ด


In [ ]:
import gdown
from pathlib import Path

# ============================================================
# แก้ไขเฉพาะลิงก์นี้
# วาง Google Drive share link ของไฟล์ checkpoint (.pth)
# ============================================================

CHECKPOINT_URL = "PASTE_YOUR_CHECKPOINT_SHARE_LINK_HERE"

CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if "PASTE_YOUR" in CHECKPOINT_URL:
    raise ValueError(
        "กรุณาใส่ Google Drive share link ของ checkpoint ในตัวแปร CHECKPOINT_URL"
    )

# ใส่ output เป็นโฟลเดอร์ เพื่อให้ gdown ใช้ชื่อไฟล์เดิมจาก Google Drive
downloaded_path = gdown.download(
    url=CHECKPOINT_URL,
    output=str(CHECKPOINT_DIR) + "/",
    quiet=False,
    fuzzy=True
)

if downloaded_path is None:
    raise RuntimeError(
        "ดาวน์โหลด checkpoint ไม่สำเร็จ กรุณาตรวจสอบลิงก์และสิทธิ์การเข้าถึงไฟล์"
    )

CHECKPOINT_PATH = Path(downloaded_path)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ไม่พบ checkpoint ที่ดาวน์โหลด: {CHECKPOINT_PATH}")

if CHECKPOINT_PATH.suffix.lower() not in {".pth", ".pt", ".ckpt"}:
    print(
        "คำเตือน: ไฟล์ checkpoint ไม่มีนามสกุล .pth, .pt หรือ .ckpt",
        CHECKPOINT_PATH.name
    )

print("Checkpoint downloaded successfully")
print("Checkpoint path:", CHECKPOINT_PATH)
print("File size:", f"{CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB")


# Session 4 — กำหนดค่าการ Predict และ Post-processing

ค่าด้านล่างเป็นค่าที่ใช้ควบคุมการทำงานของโมเดล โดยค่าเริ่มต้นถูกคงไว้ตาม workflow เดิมที่สามารถใช้งานได้แล้ว

- `PATCH_SIZE` คือขนาดภาพย่อยที่ส่งเข้าโมเดลครั้งละหนึ่ง patch
- `STRIDE` คือระยะเลื่อนของ patch หากเท่ากับ patch size จะทำงานเร็วและไม่มี overlap
- `BATCH_SIZE` คือจำนวน patch ที่ประมวลผลพร้อมกัน
- `THRESHOLD` คือเกณฑ์เปลี่ยน probability ให้เป็น building mask
- `MIN_BUILDING_AREA_M2` และ `MAX_BUILDING_AREA_M2` ใช้กรอง polygon ที่เล็กหรือใหญ่ผิดปกติ

หาก GPU memory ไม่พอ ให้ลด `BATCH_SIZE` ก่อนเป็นอันดับแรก


In [ ]:
# ============================================================
# MODEL / INFERENCE SETTINGS
# ============================================================

PATCH_SIZE = 224
STRIDE = 224
BATCH_SIZE = 8
THRESHOLD = 0.50
USE_AMP = True

# ============================================================
# VALID RGB AREA DETECTION
# ============================================================

VALID_DOWNSAMPLE = 64
VALID_MARGIN = 512
NODATA_VALUES = (0, 65535)

# ============================================================
# VECTOR POST-PROCESS SETTINGS
# ============================================================

MIN_BUILDING_AREA_M2 = 12
MAX_BUILDING_AREA_M2 = 10000
SIMPLIFY_TOLERANCE = 0.40

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Batch size:", BATCH_SIZE)
print("Threshold:", THRESHOLD)


# Session 5 — ค้นหาไฟล์ภาพจาก Google Drive

ขั้นตอนนี้แปลงค่าจาก `IMAGE_SOURCES` ให้เป็นรายการไฟล์ภาพจริงในตัวแปร `IMAGE_PATHS`

หาก source เป็นโฟลเดอร์ ระบบจะค้นหา `.tif` และ `.tiff` ทั้งในโฟลเดอร์หลักและโฟลเดอร์ย่อย จากนั้นจะลบ path ที่ซ้ำและเรียงชื่อไฟล์เพื่อให้ผลลัพธ์อ่านง่าย

Pipeline จะหยุดทันทีหากไม่พบภาพ เพื่อป้องกันการรันขั้นตอนถัดไปโดยไม่มีข้อมูล


In [ ]:
from pathlib import Path

def collect_image_paths(image_sources):
    image_paths = []

    for source in image_sources:
        source_path = Path(source)

        if source_path.is_file():
            if source_path.suffix.lower() in {".tif", ".tiff"}:
                image_paths.append(source_path)
            else:
                print(f"ข้ามไฟล์ที่ไม่ใช่ GeoTIFF: {source_path}")

        elif source_path.is_dir():
            image_paths.extend(source_path.rglob("*.tif"))
            image_paths.extend(source_path.rglob("*.tiff"))

        else:
            print(f"ไม่พบ path: {source_path}")

    # ลบรายการซ้ำและเรียงตามชื่อไฟล์
    return sorted(set(image_paths), key=lambda path: path.name.lower())


IMAGE_PATHS = collect_image_paths(IMAGE_SOURCES)

if not IMAGE_PATHS:
    raise RuntimeError(
        "ไม่พบไฟล์ .tif หรือ .tiff กรุณาตรวจสอบ IMAGE_SOURCES และ Google Drive path"
    )

print(f"พบภาพทั้งหมด {len(IMAGE_PATHS)} ภาพ")
for index, image_path in enumerate(IMAGE_PATHS, start=1):
    print(f"{index:02d}. {image_path}")


# Session 6 — ตรวจสอบข้อมูลภาพก่อนเริ่มประมวลผล

ขั้นตอนนี้อ่านเฉพาะ metadata ของแต่ละภาพ เช่น CRS, ขนาดภาพ, จำนวน band, data type, NoData และขอบเขตพิกัด

โมเดลนี้ต้องการภาพอย่างน้อย 4 bands ได้แก่ RGB และ NIR ดังนั้นโค้ดจะหยุดพร้อมข้อความแจ้งเตือนหากพบภาพที่มี band ไม่ครบ การตรวจสอบก่อนเริ่ม inference ช่วยลดโอกาสเสียเวลารันกับข้อมูลที่ไม่ตรงรูปแบบ


In [ ]:
import rasterio

# ============================================================
# QUICK INPUT CHECK
# ============================================================

for image_path in IMAGE_PATHS:
    print("=" * 80)
    print("Image:", image_path)

    with rasterio.open(image_path) as src:
        print("CRS:", src.crs)
        print("Width:", src.width)
        print("Height:", src.height)
        print("Bands:", src.count)
        print("Dtype:", src.dtypes)
        print("Nodata:", src.nodata)
        print("Bounds:", src.bounds)

        if src.count < 4:
            raise ValueError(
                f"{image_path.name} has only {src.count} band(s). "
                "This model expects 4 bands: RGB + NIR."
            )

# Session 7 — ค้นหาพื้นที่ภาพจริงและแสดง Preview

ภาพ THEOS-2 ขนาดใหญ่อาจมีพื้นที่ว่างหรือ NoData อยู่รอบภาพ หากนำพื้นที่ทั้งหมดเข้าโมเดลจะทำให้เสียเวลาและใช้หน่วยความจำโดยไม่จำเป็น

ฟังก์ชันในขั้นตอนนี้จะอ่านภาพแบบย่อเพื่อตรวจหาขอบเขตที่มีข้อมูล RGB จริง จากนั้นขยายขอบเขตเล็กน้อยด้วย margin และเก็บผลไว้ใน `VALID_BBOXES`

ระบบจะแสดง preview ของแต่ละภาพก่อน predict เพื่อให้ผู้ใช้งานตรวจสอบว่าขอบเขตที่เลือกครอบคลุมพื้นที่ภาพที่ต้องการหรือไม่


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from rasterio.windows import Window
from rasterio.enums import Resampling

# ============================================================
# VALID RGB AREA DETECTION + PREVIEW
# ============================================================

def find_valid_rgb_bbox_fast(
    image_path,
    downsample=64,
    nodata_values=(0, 65535),
    margin=512
):
    """
    หา bbox ของพื้นที่ภาพจริง โดยอ่านภาพแบบ downsample
    ไม่อ่าน full image ทั้งหมด
    """

    image_path = Path(image_path)

    with rasterio.open(image_path) as src:
        out_h = max(src.height // downsample, 1)
        out_w = max(src.width // downsample, 1)

        print("=" * 80)
        print("Finding valid RGB bbox:")
        print(image_path.name)
        print("Image size:", src.width, src.height)
        print("Downsampled size:", out_w, out_h)
        print("Metadata nodata:", src.nodata)

        # อ่านเฉพาะ RGB แบบย่อ
        rgb_small = src.read(
            [1, 2, 3],
            out_shape=(3, out_h, out_w),
            resampling=Resampling.nearest
        )

        # valid pixel = ไม่ใช่ blank ทั้ง 3 band
        valid = np.ones((out_h, out_w), dtype=bool)

        for nd in nodata_values:
            valid &= ~np.all(rgb_small == nd, axis=0)

        # กัน pixel ที่มืดสนิททั้ง 3 band
        valid &= np.any(rgb_small > 0, axis=0)

        rows, cols = np.where(valid)

        if len(rows) == 0:
            raise RuntimeError(
                f"No valid RGB area found in {image_path.name}"
            )

        left = int(cols.min() * downsample)
        right = int((cols.max() + 1) * downsample)

        top = int(rows.min() * downsample)
        bottom = int((rows.max() + 1) * downsample)

        # เพิ่ม margin กันตัดขอบภาพจริง
        left = max(left - margin, 0)
        top = max(top - margin, 0)
        right = min(right + margin, src.width)
        bottom = min(bottom + margin, src.height)

        bbox = (left, top, right, bottom)

        print("Valid RGB bbox:", bbox)
        print("Valid width:", right - left)
        print("Valid height:", bottom - top)

        return bbox


def stretch_rgb_for_display(rgb):
    rgb = rgb.astype("float32")

    # treat blank values as nan
    for nd in NODATA_VALUES:
        rgb[rgb == nd] = np.nan

    out = np.zeros_like(rgb, dtype="float32")

    for i in range(3):
        band = rgb[i]

        if np.all(np.isnan(band)):
            continue

        p2 = np.nanpercentile(band, 2)
        p98 = np.nanpercentile(band, 98)

        out[i] = np.nan_to_num(
            np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1),
            nan=0
        )

    return np.transpose(out, (1, 2, 0))


def read_rgb_preview(image_path, bbox, max_size=1200):
    left, top, right, bottom = bbox
    width = right - left
    height = bottom - top

    scale = max(width / max_size, height / max_size, 1)
    out_w = max(int(width / scale), 1)
    out_h = max(int(height / scale), 1)

    with rasterio.open(image_path) as src:
        rgb = src.read(
            [1, 2, 3],
            window=Window(left, top, width, height),
            out_shape=(3, out_h, out_w),
            resampling=Resampling.bilinear,
            boundless=False
        )

    return rgb


VALID_BBOXES = {}

for image_path in IMAGE_PATHS:
    bbox = find_valid_rgb_bbox_fast(
        image_path=image_path,
        downsample=VALID_DOWNSAMPLE,
        nodata_values=NODATA_VALUES,
        margin=VALID_MARGIN
    )

    VALID_BBOXES[Path(image_path).stem] = bbox

    rgb_preview = read_rgb_preview(
        image_path=image_path,
        bbox=bbox,
        max_size=1200
    )

    plt.figure(figsize=(9, 9))
    plt.imshow(stretch_rgb_for_display(rgb_preview))
    plt.title(
        f"{Path(image_path).name}\n"
        f"Valid RGB bbox: {bbox}"
    )
    plt.axis("off")
    plt.show()

print("VALID_BBOXES:")
VALID_BBOXES

# Session 8 — สร้างโครงสร้างโมเดล TerraMind + Local CNN

Checkpoint เก็บค่าน้ำหนักของโมเดล แต่ก่อนโหลดน้ำหนักจำเป็นต้องสร้าง architecture ให้ตรงกับตอน training

โมเดลนี้ประกอบด้วยสองส่วนหลัก:

1. **TerraMind backbone** รับข้อมูล RGB เพื่อดึง feature เชิงบริบท
2. **Local CNN branch** รับข้อมูล 4 bands ได้แก่ RGB + NIR เพื่อรักษารายละเอียดเชิงพื้นที่และขอบอาคาร

Feature จากทั้งสองส่วนจะถูกนำมารวมกันและ upsample กลับมาเป็นแผนที่ผลลัพธ์หนึ่ง channel ซึ่งแทนค่า logit ของ building footprint


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from terratorch.registry import BACKBONE_REGISTRY

# ============================================================
# MODEL ARCHITECTURE
# Must match training checkpoint
# TerraMind RGB backbone + local CNN 4-band branch
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        # IMPORTANT:
        # checkpoint was trained with "block", not "net"
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TerraMindCNNLocalBuildingSeg(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone

        # Local CNN branch uses RGB + NIR
        self.local1 = ConvBlock(4, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.local2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.local3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.local4 = ConvBlock(128, 256)

        self.tm_proj = nn.Sequential(
            nn.Conv2d(768, 256, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.fuse28 = ConvBlock(512, 256)

        self.up56 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec56 = ConvBlock(256, 128)

        self.up112 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec112 = ConvBlock(128, 64)

        self.up224 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec224 = ConvBlock(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x4):
        # TerraMind backbone uses RGB only
        rgb = x4[:, :3]

        # CNN branch uses all 4 bands
        l1 = self.local1(x4)
        l2 = self.local2(self.pool1(l1))
        l3 = self.local3(self.pool2(l2))
        l4 = self.local4(self.pool3(l3))

        feat = self.backbone({"RGB": rgb})[-1]

        b, n, c = feat.shape
        h = w = int(n ** 0.5)

        tm = feat.transpose(1, 2).reshape(b, c, h, w)
        tm = self.tm_proj(tm)

        tm = F.interpolate(
            tm,
            size=l4.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = self.fuse28(torch.cat([l4, tm], dim=1))

        x = self.dec56(torch.cat([self.up56(x), l3], dim=1))
        x = self.dec112(torch.cat([self.up112(x), l2], dim=1))
        x = self.dec224(torch.cat([self.up224(x), l1], dim=1))

        return self.out(x)


def build_model(device):
    backbone = BACKBONE_REGISTRY.build(
        "terramind_v1_base",
        pretrained=False,
        modalities=["RGB"]
    ).to(device)

    model = TerraMindCNNLocalBuildingSeg(backbone).to(device)

    return model

# Session 9 — โหลด Checkpoint เข้าสู่โมเดล

ขั้นตอนนี้เลือก GPU หาก Colab มี CUDA จากนั้นสร้างโมเดลและโหลดค่าน้ำหนักจากไฟล์ที่ดาวน์โหลดด้วย `gdown`

ใช้ `strict=True` เพื่อยืนยันว่า key ใน checkpoint ตรงกับ architecture ทุกส่วน หาก checkpoint มาจากโมเดลคนละโครงสร้าง ระบบจะหยุดและแจ้ง error แทนการรันต่อด้วยน้ำหนักที่ไม่สมบูรณ์

หลังโหลดสำเร็จ โมเดลจะถูกตั้งเป็น `eval()` เพื่อปิดพฤติกรรมที่ใช้เฉพาะระหว่าง training


In [ ]:
import torch

# ============================================================
# LOAD MODEL + CHECKPOINT
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

model = build_model(DEVICE)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("Checkpoint keys:", list(ckpt.keys())[:20])

model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

model.eval()

print("Model loaded successfully.")
print("Checkpoint:", CHECKPOINT_PATH.name)
print("Epoch:", ckpt.get("epoch"))
print("Best IoU:", ckpt.get("best_iou", ckpt.get("val_iou")))
print("Val Dice:", ckpt.get("val_dice"))

# Session 10 — ฟังก์ชัน Predict ภาพขนาดใหญ่แบบ Patch

เนื่องจากภาพ THEOS-2 มีขนาดใหญ่มาก จึงไม่สามารถส่งทั้งภาพเข้า GPU ในครั้งเดียวได้ ขั้นตอนนี้แบ่งพื้นที่ valid RGB ออกเป็น patch ขนาดเล็กและส่งเข้าโมเดลเป็น batch

ก่อนส่งแต่ละ patch เข้าโมเดลจะมีการ:

- ข้าม patch ที่เป็น NoData หรือพื้นที่ว่างเกือบทั้งหมด
- normalize แต่ละ band ด้วย percentile stretch
- ใช้ mixed precision เมื่อรันบน GPU เพื่อลดการใช้ memory
- รวม probability ของ patch กลับสู่ตำแหน่งเดิม

ผลลัพธ์จากขั้นตอนนี้คือ Probability GeoTIFF และ Raw Mask GeoTIFF


In [ ]:
import numpy as np
import rasterio
import torch
from pathlib import Path
from rasterio.windows import Window
from tqdm import tqdm
import gc

# ============================================================
# FULL VALID RGB AREA INFERENCE
# ============================================================

def make_steps(length, patch, stride):
    """
    สร้างตำแหน่ง patch ให้ครอบคลุมทั้งความยาว
    """
    if length <= patch:
        return [0]

    steps = list(range(0, length - patch + 1, stride))

    last = length - patch

    if steps[-1] != last:
        steps.append(last)

    return steps


def norm4_patch(img, nodata_values=(0, 65535)):
    """
    Normalize 4-band patch to 0-1 using percentile stretch.
    """
    img = img.astype("float32")

    for nd in nodata_values:
        img[img == nd] = np.nan

    out = np.zeros_like(img, dtype="float32")

    for b in range(img.shape[0]):
        band = img[b]

        if np.all(np.isnan(band)):
            continue

        p2 = np.nanpercentile(band, 2)
        p98 = np.nanpercentile(band, 98)

        out[b] = np.nan_to_num(
            np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1),
            nan=0
        )

    return out

def is_valid_patch(img, nodata_values=(0, 65535), min_valid_ratio=0.10):
    """
    เช็กว่า patch นี้มีข้อมูลภาพจริงพอไหม
    ถ้าเป็นดำ/NoData เยอะเกินไป จะ skip ไม่ส่งเข้า model
    """
    valid = np.ones(img.shape[1:], dtype=bool)

    for nd in nodata_values:
        valid &= ~np.all(img == nd, axis=0)

    valid &= np.any(img > 0, axis=0)

    return valid.mean() >= min_valid_ratio


def predict_full_valid_rgb_area(
    model,
    image_path,
    bbox,
    output_dir,
    patch_size=224,
    stride=224,
    batch_size=8,
    threshold=0.50,
    device="cuda",
    use_amp=True,
    min_valid_ratio=0.10
):
    """
    Predict ทั้ง valid RGB area
    และ skip patch ที่เป็น blank / NoData

    Output:
    - probability GeoTIFF
    - raw mask GeoTIFF
    """

    image_path = Path(image_path)
    output_dir = Path(output_dir)

    geotiff_dir = output_dir / "geotiff"
    geotiff_dir.mkdir(parents=True, exist_ok=True)

    left, top, right, bottom = bbox

    width = right - left
    height = bottom - top

    image_name = image_path.stem

    prob_tif = geotiff_dir / f"{image_name}_valid_rgb_probability.tif"
    raw_mask_tif = geotiff_dir / f"{image_name}_valid_rgb_raw_mask_thr{threshold}.tif"

    print("=" * 90)
    print("Full valid RGB prediction")
    print("Image:", image_path.name)
    print("BBox:", bbox)
    print("Valid RGB size:", width, height)
    print("Patch:", patch_size)
    print("Stride:", stride)
    print("Batch size:", batch_size)
    print("Threshold:", threshold)
    print("Min valid ratio:", min_valid_ratio)
    print("=" * 90)

    tmp_dir = Path("/content/tmp_full_inference")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    prob_sum_path = tmp_dir / f"{image_name}_prob_sum.dat"
    prob_count_path = tmp_dir / f"{image_name}_prob_count.dat"

    if prob_sum_path.exists():
        prob_sum_path.unlink()

    if prob_count_path.exists():
        prob_count_path.unlink()

    prob_sum = np.memmap(
        prob_sum_path,
        dtype="float32",
        mode="w+",
        shape=(height, width)
    )

    prob_count = np.memmap(
        prob_count_path,
        dtype="float32",
        mode="w+",
        shape=(height, width)
    )

    prob_sum[:] = 0
    prob_count[:] = 0

    ys = make_steps(height, patch_size, stride)
    xs = make_steps(width, patch_size, stride)

    coords = [(y, x) for y in ys for x in xs]

    print("Total possible patches:", len(coords))

    model.eval()

    skipped_patches = 0
    predicted_patches = 0

    with rasterio.open(image_path) as src:
        profile = src.profile.copy()
        out_transform = src.window_transform(
            Window(left, top, width, height)
        )

        batch_imgs = []
        batch_coords = []

        with torch.no_grad():
            for yy, xx in tqdm(coords, desc="Prepare + predict patches"):

                img = src.read(
                    [1, 2, 3, 4],
                    window=Window(
                        left + xx,
                        top + yy,
                        patch_size,
                        patch_size
                    ),
                    boundless=True,
                    fill_value=65535
                )

                # skip blank / nodata patch
                if not is_valid_patch(
                    img,
                    nodata_values=NODATA_VALUES,
                    min_valid_ratio=min_valid_ratio
                ):
                    skipped_patches += 1
                    continue

                img = norm4_patch(
                    img,
                    nodata_values=NODATA_VALUES
                )

                batch_imgs.append(img)
                batch_coords.append((yy, xx))

                if len(batch_imgs) == batch_size:
                    batch = torch.from_numpy(
                        np.stack(batch_imgs, axis=0)
                    ).float().to(device)

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                        enabled=(use_amp and str(device).startswith("cuda"))
                    ):
                        logits = model(batch)
                        probs = torch.sigmoid(logits)

                    probs_np = probs[:, 0].detach().float().cpu().numpy()

                    for j, (py, px) in enumerate(batch_coords):
                        h = min(patch_size, height - py)
                        w = min(patch_size, width - px)

                        prob_sum[py:py + h, px:px + w] += probs_np[j, :h, :w]
                        prob_count[py:py + h, px:px + w] += 1

                    predicted_patches += len(batch_imgs)

                    del batch, logits, probs, probs_np
                    torch.cuda.empty_cache()

                    batch_imgs = []
                    batch_coords = []

            # final leftover batch
            if len(batch_imgs) > 0:
                batch = torch.from_numpy(
                    np.stack(batch_imgs, axis=0)
                ).float().to(device)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=(use_amp and str(device).startswith("cuda"))
                ):
                    logits = model(batch)
                    probs = torch.sigmoid(logits)

                probs_np = probs[:, 0].detach().float().cpu().numpy()

                for j, (py, px) in enumerate(batch_coords):
                    h = min(patch_size, height - py)
                    w = min(patch_size, width - px)

                    prob_sum[py:py + h, px:px + w] += probs_np[j, :h, :w]
                    prob_count[py:py + h, px:px + w] += 1

                predicted_patches += len(batch_imgs)

                del batch, logits, probs, probs_np
                torch.cuda.empty_cache()

        print("Predicted patches:", predicted_patches)
        print("Skipped blank patches:", skipped_patches)

        prob = np.asarray(
            prob_sum / np.maximum(prob_count, 1),
            dtype="float32"
        )

        prob8 = np.clip(prob * 100, 0, 100).astype("uint8")
        raw_mask = (prob >= threshold).astype("uint8")

        prob_profile = profile.copy()
        prob_profile.update(
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype=rasterio.uint8,
            nodata=255,
            transform=out_transform,
            compress="lzw",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(prob_tif, "w", **prob_profile) as dst:
            dst.write(prob8, 1)

        mask_profile = prob_profile.copy()
        mask_profile.update(
            dtype=rasterio.uint8,
            nodata=0
        )

        with rasterio.open(raw_mask_tif, "w", **mask_profile) as dst:
            dst.write(raw_mask, 1)

    prob_sum._mmap.close()
    prob_count._mmap.close()

    gc.collect()
    torch.cuda.empty_cache()

    print("Saved probability:", prob_tif)
    print("Saved raw mask:", raw_mask_tif)

    return prob_tif, raw_mask_tif

# Session 11 — Post-processing และแปลง Mask เป็น Polygon

Raw mask จากโมเดลอาจมีจุดรบกวน รูเล็ก หรือขอบที่ไม่ต่อเนื่อง ขั้นตอนนี้จึงทำความสะอาดผลลัพธ์ก่อนแปลงเป็น vector

กระบวนการหลักประกอบด้วย:

- ลบวัตถุที่มีขนาดเล็กเกินไป
- เติมรูภายในอาคาร
- ใช้ morphological closing และ opening เพื่อลดรอยขาดและ noise
- แปลง raster mask เป็น polygon
- กรอง polygon ตามพื้นที่ขั้นต่ำและสูงสุด
- simplify geometry เล็กน้อยเพื่อให้ไฟล์ใช้งานใน QGIS ได้สะดวกขึ้น

ผลลัพธ์สุดท้ายถูกบันทึกเป็น Clean Mask GeoTIFF และ Building Footprint GeoPackage


In [ ]:
import numpy as np
import rasterio
import geopandas as gpd
import fiona
from pathlib import Path

from rasterio.features import shapes
from shapely.geometry import shape, Polygon, MultiPolygon
from skimage.morphology import (
    remove_small_objects,
    remove_small_holes,
    closing,
    opening,
    disk
)

# ============================================================
# POST-PROCESS FULL MASK TO VECTOR
# ============================================================

def fix_geom(g):
    if g is None or g.is_empty:
        return None

    try:
        g = g.buffer(0)
    except Exception:
        return None

    if g is None or g.is_empty:
        return None

    return g


def one_poly(g):
    g = fix_geom(g)

    if g is None:
        return None

    if isinstance(g, MultiPolygon):
        g = max(list(g.geoms), key=lambda x: x.area)

    if isinstance(g, Polygon):
        return g

    return None


def write_empty_gpkg(gpkg_path, crs):
    schema = {
        "geometry": "Polygon",
        "properties": {
            "building_id": "int",
            "area_m2": "float"
        }
    }

    kwargs = {
        "driver": "GPKG",
        "schema": schema,
        "layer": "buildings"
    }

    if crs is not None:
        kwargs["crs_wkt"] = crs.to_wkt()

    with fiona.open(gpkg_path, "w", **kwargs):
        pass


def postprocess_full_mask_to_vector(
    raw_mask_tif,
    output_dir,
    name,
    min_area_m2=12,
    max_area_m2=10000,
    simplify_tolerance=0.40
):
    output_dir = Path(output_dir)

    geotiff_dir = output_dir / "geotiff"
    vector_dir = output_dir / "vector"

    geotiff_dir.mkdir(parents=True, exist_ok=True)
    vector_dir.mkdir(parents=True, exist_ok=True)

    clean_mask_tif = geotiff_dir / f"{name}_valid_rgb_clean_mask.tif"
    final_gpkg = vector_dir / f"{name}_valid_rgb_building_footprints.gpkg"

    with rasterio.open(raw_mask_tif) as src:
        raw = src.read(1).astype(bool)
        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs

    print("Raw mask pixels:", int(raw.sum()))

    if raw.sum() == 0:
        clean = raw.astype("uint8")
    else:
        clean = remove_small_objects(
            raw,
            min_size=int(min_area_m2 / 0.25)
        )

        clean = remove_small_holes(
            clean,
            area_threshold=120
        )

        clean = closing(clean, footprint=disk(1))
        clean = opening(clean, footprint=disk(1))

        clean = clean.astype("uint8")

    clean_profile = profile.copy()
    clean_profile.update(
        count=1,
        dtype=rasterio.uint8,
        nodata=0,
        compress="lzw",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER"
    )

    with rasterio.open(clean_mask_tif, "w", **clean_profile) as dst:
        dst.write(clean, 1)

    geoms = []
    areas = []

    if clean.sum() > 0:
        for geom, val in shapes(
            clean.astype("uint8"),
            mask=clean > 0,
            transform=transform
        ):
            if val != 1:
                continue

            poly = one_poly(shape(geom))

            if poly is None:
                continue

            area = poly.area

            if area < min_area_m2:
                continue

            if area > max_area_m2:
                continue

            poly = one_poly(
                poly.simplify(
                    simplify_tolerance,
                    preserve_topology=True
                )
            )

            if poly is None:
                continue

            geoms.append(poly)
            areas.append(poly.area)

    if len(geoms) == 0:
        print("No building polygons found. Writing empty GPKG.")
        write_empty_gpkg(final_gpkg, crs)
        building_count = 0

    else:
        gdf = gpd.GeoDataFrame(
            {
                "building_id": range(1, len(geoms) + 1),
                "area_m2": areas
            },
            geometry=geoms,
            crs=crs
        )

        gdf.to_file(
            final_gpkg,
            driver="GPKG",
            layer="buildings"
        )

        building_count = len(gdf)

    print("Saved clean mask:", clean_mask_tif)
    print("Saved vector:", final_gpkg)
    print("Buildings:", building_count)

    return clean_mask_tif, final_gpkg, building_count

# Session 12 — รัน Pipeline กับภาพทั้งหมด

ขั้นตอนนี้นำฟังก์ชันที่เตรียมไว้ทั้งหมดมารันตามลำดับกับทุกภาพใน `IMAGE_PATHS`

สำหรับแต่ละภาพ ระบบจะ:

1. เลือก valid RGB bounding box
2. predict probability และ raw mask
3. ทำ post-processing
4. สร้าง vector building footprint
5. บันทึกข้อมูลสรุปลงในรายการ `results`

เมื่อประมวลผลครบทุกภาพ ระบบจะสร้างไฟล์ CSV เพื่อสรุป path ของผลลัพธ์และจำนวนอาคารที่ตรวจพบ


In [ ]:
import pandas as pd
from pathlib import Path
import gc

# ============================================================
# RUN PIPELINE FOR ALL IMAGES
# ============================================================

results = []

for image_path in IMAGE_PATHS:
    image_path = Path(image_path)
    image_name = image_path.stem

    bbox = VALID_BBOXES[image_name]

    image_out_dir = OUTPUT_ROOT / image_name
    image_out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print("Processing:", image_path.name)
    print("#" * 100)

    prob_tif, raw_mask_tif = predict_full_valid_rgb_area(
        model=model,
        image_path=image_path,
        bbox=bbox,
        output_dir=image_out_dir,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
        device=DEVICE,
        use_amp=USE_AMP
    )

    clean_mask_tif, final_gpkg, building_count = postprocess_full_mask_to_vector(
        raw_mask_tif=raw_mask_tif,
        output_dir=image_out_dir,
        name=image_name,
        min_area_m2=MIN_BUILDING_AREA_M2,
        max_area_m2=MAX_BUILDING_AREA_M2,
        simplify_tolerance=SIMPLIFY_TOLERANCE
    )

    row = {
        "image": str(image_path),
        "bbox_left": bbox[0],
        "bbox_top": bbox[1],
        "bbox_right": bbox[2],
        "bbox_bottom": bbox[3],
        "threshold": THRESHOLD,
        "patch_size": PATCH_SIZE,
        "stride": STRIDE,
        "probability_tif": str(prob_tif),
        "raw_mask_tif": str(raw_mask_tif),
        "clean_mask_tif": str(clean_mask_tif),
        "final_gpkg": str(final_gpkg),
        "building_count": building_count
    }

    results.append(row)

    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)

summary_csv = OUTPUT_ROOT / "full_valid_rgb_inference_summary.csv"
results_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 100)
print("PIPELINE FINISHED")
print("Summary CSV:", summary_csv)
print("=" * 100)

results_df

# Session 13 — แสดงผลลัพธ์เพื่อการตรวจสอบ

ขั้นตอนนี้สร้างภาพเปรียบเทียบ 4 ส่วน ได้แก่ ภาพ RGB, probability, raw mask และ clean mask

การแสดงผลนี้มีไว้สำหรับตรวจสอบคุณภาพอย่างรวดเร็วใน Colab ส่วนการตรวจสอบตำแหน่งและ geometry อย่างละเอียดควรเปิดไฟล์ GeoTIFF และ GeoPackage ใน QGIS


In [ ]:
import matplotlib.pyplot as plt
import rasterio
import numpy as np
from pathlib import Path
from rasterio.windows import Window
from rasterio.enums import Resampling

# ============================================================
# FINAL OUTPUT VISUALIZATION
# ============================================================

def read_rgb_preview_from_bbox(image_path, bbox, max_size=1000):
    left, top, right, bottom = bbox

    width = right - left
    height = bottom - top

    scale = max(width / max_size, height / max_size, 1)

    out_w = max(int(width / scale), 1)
    out_h = max(int(height / scale), 1)

    with rasterio.open(image_path) as src:
        rgb = src.read(
            [1, 2, 3],
            window=Window(left, top, width, height),
            out_shape=(3, out_h, out_w),
            resampling=Resampling.bilinear
        )

    return rgb


def read_single_band_preview(tif_path, max_size=1000):
    with rasterio.open(tif_path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1)

        out_w = max(int(src.width / scale), 1)
        out_h = max(int(src.height / scale), 1)

        arr = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=Resampling.nearest
        )

    return arr


for row in results:
    image_path = Path(row["image"])

    bbox = (
        int(row["bbox_left"]),
        int(row["bbox_top"]),
        int(row["bbox_right"]),
        int(row["bbox_bottom"])
    )

    prob_tif = Path(row["probability_tif"])
    raw_mask_tif = Path(row["raw_mask_tif"])
    clean_mask_tif = Path(row["clean_mask_tif"])

    rgb = read_rgb_preview_from_bbox(
        image_path,
        bbox,
        max_size=1000
    )

    prob = read_single_band_preview(
        prob_tif,
        max_size=1000
    ).astype("float32")

    if prob.max() > 1:
        prob = prob / 100.0

    raw_mask = read_single_band_preview(
        raw_mask_tif,
        max_size=1000
    )

    clean_mask = read_single_band_preview(
        clean_mask_tif,
        max_size=1000
    )

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))

    axes[0].imshow(stretch_rgb_for_display(rgb))
    axes[0].set_title("Valid RGB image")
    axes[0].axis("off")

    axes[1].imshow(prob, cmap="viridis", vmin=0, vmax=max(0.1, prob.max()))
    axes[1].set_title(f"Probability\nmax={prob.max():.3f}")
    axes[1].axis("off")

    axes[2].imshow(raw_mask, cmap="gray")
    axes[2].set_title("Raw mask")
    axes[2].axis("off")

    axes[3].imshow(clean_mask, cmap="gray")
    axes[3].set_title("Clean mask")
    axes[3].axis("off")

    plt.suptitle(
        f"{image_path.name}\n"
        f"Buildings: {row['building_count']}"
    )

    plt.tight_layout()
    plt.show()

# Session 14 — สรุปตำแหน่งไฟล์ Output

Cell สุดท้ายแสดงตำแหน่งโฟลเดอร์ผลลัพธ์ ไฟล์ summary CSV และ path ของ output แต่ละประเภทสำหรับทุกภาพ เพื่อให้ผู้ใช้งานสามารถค้นหา ดาวน์โหลด หรือย้ายไฟล์ไปยัง Google Drive ได้ง่าย


In [ ]:
print("All outputs were saved under:")
print(OUTPUT_ROOT)

print("\nSummary CSV:")
print(summary_csv)

print("\nOutput files for each image:")
for row in results:
    print("=" * 80)
    print("Image:", Path(row["image"]).name)
    print("Probability GeoTIFF:", row["probability_tif"])
    print("Raw mask GeoTIFF:", row["raw_mask_tif"])
    print("Clean mask GeoTIFF:", row["clean_mask_tif"])
    print("Vector GPKG:", row["final_gpkg"])
    print("Detected buildings:", row["building_count"])
